In [1]:
!pip install bert-score
!pip install evaluate

In [ ]:
import pandas as pd
import evaluate
from bert_score import score as b_score
from sentence_transformers import SentenceTransformer, util

# Load data
df_correct = pd.read_csv("", delimiter=";", encoding="utf-8") # insert reference dataset path
df_inference = pd.read_csv("", delimiter=";", encoding="utf-8") # insert inference dataset path
df_fine = pd.read_csv("", delimiter=",", encoding="utf-8") # insert finetuned dataset path
df_rag = pd.read_csv("", delimiter=",", encoding="utf-8") # insert RAG dataset path

rouge_metric = evaluate.load("rouge")
model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def run_unified_evaluation(df_pred, df_base, model_name="Modell"):

    eval_df = pd.merge(df_pred[['id', 'answer']], 
                       df_base[['id', 'correct_answer']], 
                       on="id", 
                       how="inner")
    
    preds = eval_df['answer'].astype(str).tolist()
    refs = eval_df['correct_answer'].astype(str).tolist()
    
    # metric 1: BERTScore
    # Evaluates semantic equivalence using contextual embeddings.
    # Focus: Does the model capture the correct meaning regardless of specific wording?
    P, R, F1 = b_score(preds, refs, lang="de", verbose=False, model_type="bert-base-multilingual-cased")
    bs_f1 = F1.mean().item()
    
    # metric 2: ROUGE-L
    # Measures the Longest Common Subsequence between prediction and reference.
    # Focus: How much does the sentence structure and word order match the base dataset?
    rouge_results = rouge_metric.compute(predictions=preds, references=refs)
    rL = rouge_results['rougeL']
    
    # metric 3: Cosine Similarity
    # Measures the cosine of the angle between two sentence embeddings.
    # Focus: Is the response in the correct global thematic context?
    emb1 = model_sim.encode(preds, convert_to_tensor=True)
    emb2 = model_sim.encode(refs, convert_to_tensor=True)
    cos_sim = util.cos_sim(emb1, emb2).diag().mean().item()

    return {
        "Model": model_name,
        "BERTScore (F1)": round(bs_f1, 4),
        "ROUGE-L": round(rL, 4),
        "Cosine Sim.": round(cos_sim, 4)
    }

results = []
results.append(run_unified_evaluation(df_inference, df_correct, "Inference"))
results.append(run_unified_evaluation(df_fine, df_correct, "Finetuned"))
results.append(run_unified_evaluation(df_rag, df_correct, "RAG"))


df_final_eval = pd.DataFrame(results)
print(df_final_eval.to_string(index=False))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Model  BERTScore (F1)  ROUGE-L  Cosine Sim.
Inference          0.7262   0.1938       0.6510
Finetuned          0.7177   0.1592       0.5823
      RAG          0.6828   0.1358       0.5862
